In [2]:
import anthropic, os
from pathlib import Path  # Para manipulação de caminhos de arquivos
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
models = client.models.list()

# Ordena a lista de modelos pelo atributo 'id' e imprime ordenado
model_ids = sorted([m.id for m in models.data])
for mid in model_ids:
    print(mid)

claude-fable-5
claude-haiku-4-5-20251001
claude-opus-4-1-20250805
claude-opus-4-5-20251101
claude-opus-4-6
claude-opus-4-7
claude-opus-4-8
claude-sonnet-4-5-20250929
claude-sonnet-4-6


In [13]:
model_name = "claude-haiku-4-5-20251001"
prompt = "read the file data.csv"
response = client.messages.create(
    model=model_name,
    max_tokens=4096,
    temperature=0.0,
    messages=[{"role": "user", "content": prompt}]
)

In [18]:
response

Message(id='msg_01KmWQLAvwx8TcTsLwThp32i', container=None, content=[TextBlock(citations=None, text="I don't have the ability to directly access files on your system. However, I can help you read and analyze a CSV file in a few ways:\n\n1. **Share the contents**: Copy and paste the contents of `data.csv` here, and I can help you analyze it.\n\n2. **Provide code**: If you want code to read it, here are some options:\n\n**Python (using pandas):**\n```python\nimport pandas as pd\ndf = pd.read_csv('data.csv')\nprint(df)\n```\n\n**Python (using csv module):**\n```python\nimport csv\nwith open('data.csv', 'r') as file:\n    reader = csv.reader(file)\n    for row in reader:\n        print(row)\n```\n\n**Command line:**\n```bash\ncat data.csv\n```\n\nWhich approach would work best for you? Or feel free to paste the CSV content and I'll help analyze it!", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='messa

In [15]:
import json
print(json.dumps(response[0], indent=4, ensure_ascii=False))

TypeError: 'Message' object is not subscriptable

### tool tests

In [12]:
from agents.tools import to_anthropic_tools
import json
from pprint import pprint
tool_names = ["view_file", "search_files"]
anthropic_tools = to_anthropic_tools(tool_names)

print(json.dumps(anthropic_tools, indent=4, ensure_ascii=False))

[
    {
        "name": "view_file",
        "description": "Read file contents to understand data structure, columns, format. Use this to inspect CSV files, understand data schemas, check date ranges.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "Relative path to file in workspace (e.g., 'data.csv', 'uploads/sales.csv')"
                }
            },
            "required": [
                "path"
            ]
        }
    },
    {
        "name": "search_files",
        "description": "List files matching a pattern in workspace. Use to discover available data files.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pattern": {
                    "type": "string",
                    "description": "Glob pattern (e.g., '*.csv', '*.xlsx', 'uploads/*')"
                }
            },
    

## Testing agents 

### imports

In [82]:
from agents.planner_agent import planner_agent_file
from agents.helper_contet_agent import helper_contet_agent, _extract_json

def _extract_input_files(data_source_meta: dict) -> str:
    """
    Extract input_files string from data_source_meta for planner agent.

    Args:
        data_source_meta: Dict with csv_path, csv_paths, or database_id

    Returns:
        String representation of input files
    """
    # Support both csv_paths (list) and csv_path (single string) for backward compat
    if "csv_paths" in data_source_meta:
        paths = data_source_meta["csv_paths"]
        if isinstance(paths, list) and paths:
            return ", ".join(paths)
        return str(paths)
    elif "csv_path" in data_source_meta:
        return data_source_meta["csv_path"]
    elif "database_id" in data_source_meta:
        return f"database:{data_source_meta['database_id']}"
    return ""

### Planner agent

In [83]:
input_files = _extract_input_files(
    r"C:\Users\Leal\AI_Projects\time_series_analysis_agent\workspace\uploads"
)

raw_response, output = planner_agent_file(
    prompt='helpe me to analyse my data, Understandind trends over time',
    conversation_history='',
    input_files=input_files
)

step_list = output['model_answer']['plan']

In [85]:
print(raw_response)
print(output)
print(output['model_answer']['plan'])
for number, step in enumerate(step_list):
    n = number+1
    print(f'{n} step: {step}')

None
{'prompt': 'helpe me to analyse my data, Understandind trends over time', 'model_answer': {'answer': "I've found your data.csv file with 31 rows of time-series data (dates and values from January 2023). I've created a comprehensive plan to analyze trends over time. This will include data validation, trend analysis, visualizations, and statistical insights.", 'plan': [{'description': 'Load and validate the data: Parse dates, check for missing values, and verify data types', 'reasoning': 'Essential first step to ensure data quality and proper date handling for time-series analysis', 'thinking_budget': None}, {'description': 'Calculate descriptive statistics: Mean, median, standard deviation, min/max values, and identify the date range', 'reasoning': 'Simple statistical calculations to understand the baseline characteristics of your data', 'thinking_budget': None}, {'description': 'Perform trend analysis: Calculate moving averages (7-day and 30-day) and identify overall trend directi

### Test helper agent

In [59]:
# Run helper with the plan that includes rationale

result = helper_contet_agent(step_list)

In [60]:
# Print results with rationale
print(result)
print("Helper agent output with rationale:\n")
for i in result:
    print(i)
    print()

[{'step_number': 1, 'description': 'Load and validate the data: Parse the CSV file, convert date column to datetime format, check for missing values and data quality issues', 'rationale': 'Ensures data is properly formatted for time-series analysis and identifies any data quality issues that could affect trend analysis', 'skill_needed': False, 'arg': None}, {'step_number': 2, 'description': 'Calculate basic time-series statistics: Compute mean, median, standard deviation, min/max values, and identify the overall trend direction', 'rationale': "Provides a quantitative baseline understanding of the data's central tendency, variability, and range", 'skill_needed': False, 'arg': None}, {'step_number': 3, 'description': 'Perform trend analysis: Calculate linear regression slope, moving averages (7-day and 14-day), and identify upward/downward trends', 'rationale': 'Reveals underlying patterns and smooths out daily fluctuations to show the true direction of change over time', 'skill_needed':

### Test Executor Context Builder Agent

In [87]:
from agents.executor_context_builder import build_context, ExecutionContextPackage

In [88]:
# Test build_context with mock helper agent output (no planner cells required)

mock_plan = [
    {"step_number": 1, "description": "Load and validate the data: Parse the CSV file, convert date column to datetime format, check for missing values and data quality issues", "rationale": "Ensures data is properly formatted for time-series analysis and identifies any data quality issues that could affect trend analysis", "skill_needed": False, "arg": None},
    {"step_number": 2, "description": "Calculate basic time-series statistics: Compute mean, median, standard deviation, min/max values, and identify the overall trend direction", "rationale": "Provides a quantitative baseline understanding of the data's central tendency, variability, and range", "skill_needed": False, "arg": None},
    {"step_number": 3, "description": "Perform trend analysis: Calculate linear regression slope, moving averages (7-day and 14-day), and identify upward/downward trends", "rationale": "Reveals underlying patterns and smooths out daily fluctuations to show the true direction of change over time", "skill_needed": True, "arg": "C:/Users/Leal/AI_Projects/time_series_analysis_agent/skills/analyzing-time-series/SKILL.md"},
    {"step_number": 4, "description": "Detect seasonality and cyclical patterns: Analyze day-to-day changes, identify peaks and troughs, and look for repeating patterns", "rationale": "Uncovers if there are regular patterns or cycles in your data that repeat over specific time intervals", "skill_needed": True, "arg": "C:/Users/Leal/AI_Projects/time_series_analysis_agent/skills/analyzing-time-series/SKILL.md"},
    {"step_number": 5, "description": "Create visualizations: Generate line plots with trend lines, moving averages, and highlight key turning points", "rationale": "Visual representations make trends immediately apparent and easier to communicate to stakeholders", "skill_needed": False, "arg": None},
    {"step_number": 6, "description": "Calculate period-over-period growth rates: Compute daily, weekly changes and percentage growth to quantify trend magnitude", "rationale": "Provides concrete metrics showing how fast values are changing and enables comparison across different time periods", "skill_needed": False, "arg": None},
    {"step_number": 7, "description": "Generate summary report: Document key findings, trend direction, volatility assessment, and provide actionable insights", "rationale": "Consolidates all analysis into a clear narrative that explains what the trends mean and what they might indicate", "skill_needed": True, "arg": "C:/Users/Leal/AI_Projects/time_series_analysis_agent/skills/docx/SKILL.md"},
]

def _load_skill_context(skill_path: str | None) -> str | None:
    if not skill_path:
        return None
    path = Path(skill_path)
    if not path.exists():
        return None
    content = path.read_text(encoding="utf-8")
    lines = content.splitlines()
    if lines and lines[0].strip() == "---":
        for i, line in enumerate(lines[1:], start=1):
            if line.strip() == "---":
                return "\n".join(lines[i + 1:])
    return content

def _helper_step_for_context(helper_step: dict) -> dict:
    """Map helper output fields to what build_context expects."""
    return {
        "description": helper_step.get("description", ""),
        "reasoning": helper_step.get("rationale", helper_step.get("reasoning", "")),
        "thinking_budget": helper_step.get("thinking_budget"),
        "output_type": helper_step.get("output_type", "card"),
    }

# Change current_step to test a different step (1-7); use 3+ to test skill loading
current_step = 1
helper_step = mock_plan[current_step - 1]
step_for_context = _helper_step_for_context(helper_step)

skill_context = None
if helper_step.get("skill_needed") and helper_step.get("arg"):
    skill_context = _load_skill_context(helper_step["arg"])

ctx = build_context(
    task_id="test_task_id",
    output_name="test_output",
    step=step_for_context,
    csv_paths=["data.csv"],
    current_step=current_step,
    skill_context=skill_context,
)

print(f"Testing step {current_step}: {helper_step['description'][:60]}...")
print("\nExecutionContextPackage:")
print(ctx)
print("\n--- Prompt text (for code generator) ---\n")
print(ctx.to_prompt_text())

Testing step 1: Load and validate the data: Parse the CSV file, convert date...

ExecutionContextPackage:
ExecutionContextPackage(step={'step_number': 1, 'description': 'Load and validate the data: Parse the CSV file, convert date column to datetime format, check for missing values and data quality issues', 'reasoning': 'Ensures data is properly formatted for time-series analysis and identifies any data quality issues that could affect trend analysis', 'thinking_budget': None, 'output_type': 'card'}, workspace={'workspace_root': 'workspace/', 'input_files': ['data.csv'], 'output_dir': 'outputs/test_task_id/step_1', 'ui_json_path': 'outputs/test_task_id/step_1/test_output_ui.json', 'script_path': 'outputs/test_task_id/step_1/script.py'}, data_preview={'files': [{'path': 'data.csv', 'row_count': 30, 'columns': [{'name': 'date', 'dtype': 'str', 'sample': '2023-01-01'}, {'name': 'value', 'dtype': 'int64', 'sample': '100'}], 'null_counts': {'date': 0, 'value': 0}, 'date_range': {'column': '

### executor code generator

In [89]:
# Test generate_code using ctx from the build_context cell above
# Requires: cell 0 (load_dotenv), cell 13 (imports), cell 14 (ctx)

from agents.executor_code_generator import generate_code, CodeGenerationResult

code_result = generate_code(
    context=ctx,
    model="claude-haiku-4-5-20251001",
    verbose=True,
)

print(f"success: {code_result.success}")
print(f"input_tokens: {code_result.input_tokens}")
print(f"output_tokens: {code_result.output_tokens}")
if code_result.error:
    print(f"error: {code_result.error}")

if code_result.success:
    print("\n--- Generated script ---\n")
    print(code_result.code)
else:
    print("\n--- Raw LLM response ---\n")
    print(code_result.raw_response)


👤 USER: Generate a Python script for this analysis step.

## STEP
{
  "step_number": 1,
  "description": "Load and validate the data: Parse the CSV file, convert date column to datetime format, check for missing values and data quality issues",
  "reasoning": "Ensures data is properly formatted for time-series analysis and identifies any data quality issues that could affect trend analysis",
  "thinking_budget": null,
  "output_type": "card"
}

## WORKSPACE PATHS
{
  "workspace_root": "workspace/",
  "input_files": [
    "data.csv"
  ],
  "output_dir": "outputs/test_task_id/step_1",
  "ui_json_path": "outputs/test_task_id/step_1/test_output_ui.json",
  "script_path": "outputs/test_task_id/step_1/script.py"
}

## DATA PREVIEW
{
  "files": [
    {
      "path": "data.csv",
      "row_count": 30,
      "columns": [
        {
          "name": "date",
          "dtype": "str",
          "sample": "2023-01-01"
        },
        {
          "name": "value",
          "dtype": "int64",
    

In [90]:
# Test run_code using code_result + ctx from cells above
# Requires: cell 14 (ctx), code generator cell (code_result)

from agents.executor_code_runner import run_code, CodeRunResult
import json

if not code_result.success:
    raise RuntimeError(f"Code generation failed: {code_result.error}")

run_result = run_code(
    code=code_result.code,
    script_path=ctx.workspace["script_path"],
    ui_json_path=ctx.workspace["ui_json_path"],
    timeout=60,
)

print(f"success: {run_result.success}")
print(f"returncode: {run_result.returncode}")
print(f"ui_json_valid: {run_result.ui_json_valid}")
print(f"generated_files: {run_result.generated_files}")
if run_result.error:
    print(f"error: {run_result.error}")
if run_result.stdout:
    print(f"\n--- stdout ---\n{run_result.stdout}")
if run_result.stderr:
    print(f"\n--- stderr ---\n{run_result.stderr}")

if run_result.success:
    ui_path = Path("workspace") / ctx.workspace["ui_json_path"]
    ui_data = json.loads(ui_path.read_text(encoding="utf-8"))
    print("\n--- ui.json preview ---\n")
    print(json.dumps(ui_data, indent=2, ensure_ascii=False)[:3000])

[CODE_RUNNER] Script saved to: C:\Users\Leal\AI_Projects\time_series_analysis_agent\workspace\outputs\test_task_id\step_1\script.py
[CODE_RUNNER] Executing script with timeout=60s...
[CODE_RUNNER] Script finished with returncode=1
[STDOUT] ✗ Error: File not found - [Errno 2] No such file or directory: 'workspace/data.csv'

success: False
returncode: 1
ui_json_valid: False
generated_files: ['outputs\\test_task_id\\step_1\\script.py']
error: Script exited with code 1

--- stdout ---
✗ Error: File not found - [Errno 2] No such file or directory: 'workspace/data.csv'



### executor orchestrator

In [ ]:
# Test executor_orchestrator end-to-end (context → codegen → run → retry)
# Requires: cell 0 (load_dotenv), cell 14 (mock_plan + _load_skill_context)

import json
import pprint

from main import SessionLocal, Task
from agents.executor_orchestrator import executor_orchestrator

ORCH_TASK_ID = "test_task_id"
ORCH_OUTPUT_NAME = "test_output"
ORCH_CURRENT_STEP = 1  # change 1-7 to test other steps


def _helper_to_executor_step(helper_step: dict) -> dict:
    """Map helper output to the step format expected by the orchestrator."""
    step = {
        "description": helper_step.get("description", ""),
        "reasoning": helper_step.get("rationale", helper_step.get("reasoning", "")),
        "thinking_budget": helper_step.get("thinking_budget"),
        "output_type": helper_step.get("output_type", "card"),
    }
    if helper_step.get("skill_needed") and helper_step.get("arg"):
        step["skill_context"] = _load_skill_context(helper_step["arg"])
    return step


orchestrator_steps = [_helper_to_executor_step(s) for s in mock_plan]

db = SessionLocal()
try:
    task = db.query(Task).filter(Task.id == ORCH_TASK_ID).first()
    if task is None:
        task = Task(
            id=ORCH_TASK_ID,
            prompt="Notebook orchestrator test",
            status="executing",
            data_source_type="csv",
            data_source_meta=json.dumps({"csv_path": "data.csv"}),
        )
        db.add(task)
    else:
        task.data_source_meta = json.dumps({"csv_path": "data.csv"})
        task.status = "executing"
    db.commit()

    print(f"Running orchestrator: step {ORCH_CURRENT_STEP}/{len(orchestrator_steps)}")
    orch_result = executor_orchestrator(
        task_id=ORCH_TASK_ID,
        output_name=ORCH_OUTPUT_NAME,
        steps=orchestrator_steps,
        current_step=ORCH_CURRENT_STEP,
        db=db,
    )
finally:
    db.close()

print("\n--- StepExecutionResult ---")
pprint.pprint(orch_result)

OperationalError: (psycopg2.OperationalError) could not translate host name "db" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)